In [2]:
EMBED_MODEL = "embeddinggemma:latest"   
#EMBED_MODEL = "nomic-embed-text"

WORDS_PER_CHUNK = 300
OVERLAP_WORDS   = 60
TOPK            = 5

# Toggle downloads off if you want to stay strictly offline (then we'll use tiny built-in samples).
DOWNLOAD_FROM_WEB = True

# A small selection of long classics (public domain). Even 2–3 will give you 300+ chunks.

GUTENBERG_BOOKS = {
    #"Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    #"Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    #"Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
    #"Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    #"Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
    #"A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
    #"The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    #"Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    #"Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
    #"The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
    #"Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
    #"Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",

}


GUTENBERG = [
    (title, url) for title, url in GUTENBERG_BOOKS.items()
]

CORPUS_DIR = "corpus_jupyter"

In [3]:
import os, re, json, textwrap
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm

import ollama
import faiss


# Minimal helper: strip Project Gutenberg boilerplate if found.
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

def strip_gutenberg_boilerplate(txt: str) -> str:
    start = START_MARK.search(txt)
    end = END_MARK.search(txt)
    if start and end and end.start() > start.end():
        return txt[start.end():end.start()].strip()
    # Fallback heuristics
    txt = re.sub(r"(?s)^.*?Project Gutenberg.*?eBook.*?\n", "", txt, flags=re.I)
    txt = re.sub(r"(?s)End of Project Gutenberg.*$", "", txt, flags=re.I)
    return txt.strip()

def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

In [4]:
# === Check if pre-built artifacts exist; if so, load and skip heavy work ===
import json as _json

#ARTIFACTS_DIR = "C:\\Users\\arunk\\OneDrive\\Documents\\GenAI - Scaler\\LLM\\RAG\\rag_artifacts"
ARTIFACTS_DIR = "rag_artifacts"

_artifact_files = ["chunks.json", "embeddings.npy", "faiss_index.bin", "config.json"]

ARTIFACTS_LOADED = all((Path(ARTIFACTS_DIR) / f).exists() for f in _artifact_files)

if ARTIFACTS_LOADED:
    print("✅ Found pre-built artifacts! Loading from disk …")
    with open(Path(ARTIFACTS_DIR) / "chunks.json", encoding="utf-8") as _f:
        chunks = _json.load(_f)
    emb = np.load(str(Path(ARTIFACTS_DIR) / "embeddings.npy"))
    index = faiss.read_index(str(Path(ARTIFACTS_DIR) / "faiss_index.bin"))
    print(f"   {len(chunks)} chunks | embeddings {emb.shape} | FAISS {index.ntotal} vectors")
    print("   Skipping download, chunking, embedding, and index-build cells.")
else:
    print("⏳ Artifacts not found — will build from scratch.")

⏳ Artifacts not found — will build from scratch.


In [5]:
if not ARTIFACTS_LOADED:
    Path(CORPUS_DIR).mkdir(parents=True, exist_ok=True)

    docs = []  # list of dicts: {title, text, path}

    if DOWNLOAD_FROM_WEB:
        for title, url in GUTENBERG:
            out_path = Path(CORPUS_DIR) / f"{title.replace(' ', '_')}.txt"
            if not out_path.exists():
                print(f"Downloading: {title}")
                try:
                    r = requests.get(url, timeout=60)
                    r.raise_for_status()
                    clean = strip_gutenberg_boilerplate(r.text)
                    out_path.write_text(clean, encoding="utf-8")
                except Exception as e:
                    print(f"  Failed ({e}); skipping.")
            else:
                print(f"Exists: {out_path.name}")
            if out_path.exists():
                docs.append({
                    "title": title,
                    "text": out_path.read_text(encoding="utf-8", errors="ignore"),
                    "path": str(out_path)
                })

    print(f"Loaded {len(docs)} docs")
else:
    print("⏩ Skipped (artifacts already loaded)")

Exists: War_and_Peace.txt
Loaded 1 docs


In [6]:
if not ARTIFACTS_LOADED:
    # print no of words in each document
    for d in docs:
        num_words = len(d["text"].split())
        print(f"{d['title']}: {num_words} words")
else:
    print("⏩ Skipped (artifacts already loaded)")

War and Peace: 563286 words


In [7]:
if not ARTIFACTS_LOADED:
    # --- Chunking ---
    chunks = []  # list of dicts: {id, title, text, preview, source_path, chunk_index}
    for d in docs:
        words = d["text"].split()
        if not words:
            continue
        step = max(1, WORDS_PER_CHUNK - OVERLAP_WORDS)
        idx = 0
        chunk_i = 0
        while idx < len(words):
            segment = words[idx:idx+WORDS_PER_CHUNK]
            if len(segment) < max(60, WORDS_PER_CHUNK//4):
                break
            text_seg = " ".join(segment)
            chunks.append({
                "id": f"{Path(d['title']).name.replace(' ', '_')}#chunk{chunk_i}",
                "title": d["title"],
                "text": text_seg,
                "preview": text_seg[:400],
                "source_path": d["path"],
                "chunk_index": chunk_i,
            })
            chunk_i += 1
            idx += step

    print(f"Total chunks: {len(chunks)}")
    pd.DataFrame([
        {"id": c["id"], "title": c["title"], "preview": c["preview"][:140] + ("…" if len(c["preview"])>140 else "")}
        for c in chunks[:8]
    ])
else:
    print(f"⏩ Skipped — {len(chunks)} chunks already loaded from artifacts")

Total chunks: 2347


In [8]:
chunks[1000]

{'id': 'War_and_Peace#chunk1000',
 'title': 'War and Peace',
 'text': 'sorry for everyone, for myself, and for everyone. And I was innocent—that was the chief thing,” said Natásha. “Do you remember?” “I remember,” answered Nicholas. “I remember that I came to you afterwards and wanted to comfort you, but do you know, I felt ashamed to. We were terribly absurd. I had a funny doll then and wanted to give it to you. Do you remember?” “And do you remember,” Natásha asked with a pensive smile, “how once, long, long ago, when we were quite little, Uncle called us into the study—that was in the old house—and it was dark—we went in and suddenly there stood...” “A Negro,” chimed in Nicholas with a smile of delight. “Of course I remember. Even now I don’t know whether there really was a Negro, or if we only dreamed it or were told about him.” “He was gray, you remember, and had white teeth, and stood and looked at us....” “Sónya, do you remember?” asked Nicholas. “Yes, yes, I do remember somethi

In [9]:
if not ARTIFACTS_LOADED:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import time
    import numpy as np
    from tqdm import tqdm
    import ollama
    import faiss

    # Tune this based on your machine; 4–8 is usually the sweet spot.
    MAX_WORKERS = 6
    RETRIES = 3
    BACKOFF = 0.6  # seconds, linear backoff

    def embed_text(text: str) -> np.ndarray:
        last_err = None
        for attempt in range(1, RETRIES + 1):
            try:
                resp = ollama.embeddings(model=EMBED_MODEL, prompt=text)
                return np.asarray(resp["embedding"], dtype="float32")
            except Exception as e:
                last_err = e
                if attempt < RETRIES:
                    time.sleep(BACKOFF * attempt)
        raise last_err

    # Parallelize over chunks while preserving original order
    emb_vectors = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(embed_text, c["text"]): i for i, c in enumerate(chunks)}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Embedding ({EMBED_MODEL})"):
            i = futures[fut]
            emb_vectors[i] = fut.result()
else:
    print(f"⏩ Skipped — embeddings already loaded from artifacts")

Embedding (embeddinggemma:latest): 100%|██████████| 2347/2347 [43:23<00:00,  1.11s/it] 


In [10]:
if not ARTIFACTS_LOADED:
    emb = np.vstack(emb_vectors)  # shape (N, d)
    d = emb.shape[1]
    print("Embeddings shape:", emb.shape)

    # L2-normalize and build FAISS IP index (IP on unit-norm == cosine similarity)
    emb = l2_normalize(emb)
    index = faiss.IndexFlatIP(d)
    index.add(emb)
    print("FAISS index size:", index.ntotal)
else:
    print(f"⏩ Skipped — FAISS index already loaded ({index.ntotal} vectors)")

Embeddings shape: (2347, 768)
FAISS index size: 2347


In [11]:
if not ARTIFACTS_LOADED:
    # === Save all time-consuming artifacts to disk for fast Flask deployment ===
    import json, pickle

    ARTIFACTS_DIR = "rag_artifacts"
    Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

    # 1. Save chunks metadata as JSON
    with open(Path(ARTIFACTS_DIR) / "chunks.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)
    print(f"Saved {len(chunks)} chunks to {ARTIFACTS_DIR}/chunks.json")

    # 2. Save normalized embeddings as numpy array
    np.save(Path(ARTIFACTS_DIR) / "embeddings.npy", emb)
    print(f"Saved embeddings shape {emb.shape} to {ARTIFACTS_DIR}/embeddings.npy")

    # 3. Save FAISS index
    faiss.write_index(index, str(Path(ARTIFACTS_DIR) / "faiss_index.bin"))
    print(f"Saved FAISS index ({index.ntotal} vectors) to {ARTIFACTS_DIR}/faiss_index.bin")

    # 4. Save config so Flask app knows model name, chunk params, etc.
    config = {
        "EMBED_MODEL": EMBED_MODEL,
        "WORDS_PER_CHUNK": WORDS_PER_CHUNK,
        "OVERLAP_WORDS": OVERLAP_WORDS,
        "TOPK": TOPK,
    }
    with open(Path(ARTIFACTS_DIR) / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    print(f"Saved config to {ARTIFACTS_DIR}/config.json")

    print("\n✅ All artifacts saved!")
else:
    print("⏩ Skipped — artifacts already exist on disk")

Saved 2347 chunks to rag_artifacts/chunks.json
Saved embeddings shape (2347, 768) to rag_artifacts/embeddings.npy
Saved FAISS index (2347 vectors) to rag_artifacts/faiss_index.bin
Saved config to rag_artifacts/config.json

✅ All artifacts saved!


In [12]:
# --- Simplest windowed retrieval: no decay, preserve base-hit order ---

# Lookup map for (doc, chunk_index) -> global idx
# Create a dictionary mapping (source_path, chunk_index) tuples to their global index
KEY_TO_IDX = {(c["source_path"], c["chunk_index"]): gi for gi, c in enumerate(chunks)}

#`I` and `D` are the standard outputs from FAISS index search:

#- **`I`** (Indices): A numpy array containing the **indices** of the nearest neighbors found by the ANN search. Shape is `(num_queries, topk)`. In your case, `I[0]` contains the global chunk indices of the top-k most similar chunks.

#- **`D`** (Distances): A numpy array containing the **similarity scores** (distances) for each neighbor. Shape is `(num_queries, topk)`. In your case, `D[0]` contains the cosine similarity scores corresponding to each chunk in `I[0]`.

#In the context of your code:
#- `I[0]` gives you which chunks matched (by their global index in the `chunks` list)
#- `D[0]` gives you how similar each match is (higher scores = better matches)

#Both arrays are ordered by relevance, with the best match first.

def expand_with_neighbors_simple(I, D, chunks, neighbors=1, max_out=8):
    """
    For the initial ANN hits (I, D), walk hits in rank order and
    append each hit plus its ±neighbors (within the same document).
    No scoring/decay; just preserve base-hit order and dedupe.
    Then merge contiguous chunks into spans and return merged contexts.
    """
    # Initialize a set to track chunks we've already included
    seen = set()
    # List to maintain the order of selected chunk indices
    ordered_idxs = []

    # Walk initial hits in order of relevance
    for gi in I[0]:
        # Get the chunk metadata for this global index
        c = chunks[int(gi)]
        doc = c["source_path"]
        ci  = c["chunk_index"]
        
        # Iterate through the hit and its neighboring chunks (e.g., ±1 or ±2)
        for delta in range(-neighbors, neighbors + 1):
            # Calculate neighbor's key
            key = (doc, ci + delta)
            # Look up the global index for this neighbor
            j = KEY_TO_IDX.get(key)
            
            # Skip if neighbor doesn't exist (e.g., at document boundaries)
            if j is None:
                continue
            
            # Add neighbor to our list if we haven't seen it yet
            if j not in seen:
                ordered_idxs.append(j)
                seen.add(j)
            
            # Stop if we've reached the maximum output limit
            if len(ordered_idxs) >= max_out:
                break
        
        # Break outer loop if we've reached the maximum output limit
        if len(ordered_idxs) >= max_out:
            break

    # Sort selected chunks by (document, chunk_index) to prepare for merging contiguous chunks
    ordered_idxs.sort(key=lambda j: (chunks[j]["source_path"], chunks[j]["chunk_index"]))

    # Merge contiguous chunks from the same document into spans
    merged = []
    span = None
    
    for gi in ordered_idxs:
        c = chunks[gi]
        
        # Check if this chunk is contiguous with the current span
        if (span 
            and c["source_path"] == span["source_path"] 
            and c["chunk_index"] == span["end_ci"] + 1):
            # Extend the current span
            span["end_ci"] += 1
            span["idxs"].append(gi)
        else:
            # Save the previous span if it exists
            if span: 
                merged.append(span)
            
            # Start a new span
            span = {
                "title": c["title"],
                "source_path": c["source_path"],
                "start_ci": c["chunk_index"],
                "end_ci": c["chunk_index"],
                "idxs": [gi],
            }
    
    # Don't forget to add the last span
    if span: 
        merged.append(span)

    # Build final context objects by concatenating text from merged spans
    contexts = []
    for s in merged:
        # Join all chunk texts in this span
        text = "\n\n".join(chunks[i]["text"] for i in s["idxs"])
        
        # Create context object with metadata
        contexts.append({
            "title": s["title"],
            "source_path": s["source_path"],
            "start_chunk": s["start_ci"],
            "end_chunk": s["end_ci"],
            "text": text,
            "approx_words": sum(len(chunks[i]["text"].split()) for i in s["idxs"]),
            "span_count": len(s["idxs"]),
        })
    
    return contexts


def search_windowed(query: str, topk: int = TOPK, max_out: int = 8):
    """
    1) ANN search (topk)
    2) Choose neighbor width (based on chunk size)
    3) Expand with simple ±neighbors (no decay) and merge
    Returns: (df_hits, contexts)
    """
    # Generate embedding for the query
    q_vec = np.asarray(ollama.embeddings(model=EMBED_MODEL, prompt=query)["embedding"], dtype="float32")
    # Normalize the query vector
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-12)
    
    # Perform ANN search to find top-k nearest neighbors
    D, I = index.search(q_vec.reshape(1, -1), topk)

    # Heuristic: decide neighbor window size based on chunk size
    # Larger chunks need fewer neighbors; smaller chunks may need more
    neighbors = 1 if WORDS_PER_CHUNK >= 260 else 2

    # Heuristic: decide how many base hits to expand based on score margin
    top1 = float(D[0][0])  # Best match score
    top2 = float(D[0][1]) if len(D[0]) > 1 else 0.0  # Second-best score
    margin = top1 - top2  # Gap between top two matches
    
    # If top match is strong and clearly better than second, expand only 1 hit
    # Otherwise, expand up to 3 hits
    init_topk = 1 if (top1 >= 0.35 and margin >= 0.05) else min(3, topk)

    # Slice arrays to only include the hits we want to expand
    D_init = np.array([D[0][:init_topk]])
    I_init = np.array([I[0][:init_topk]])

    # Expand hits with neighbors and merge into context spans
    contexts = expand_with_neighbors_simple(I_init, D_init, chunks, neighbors=neighbors, max_out=max_out)

    # Build a DataFrame to display all ANN hits (for debugging/visibility)
    rows = []
    for score, idx_ in zip(D[0].tolist(), I[0].tolist()):
        m = chunks[idx_]
        rows.append({
            "score": round(score, 3),
            "id": m["id"],
            "title": m["title"],
            "source_path": m["source_path"],
            "chunk_index": m["chunk_index"],
            "preview": (m["preview"][:220] + "…") if len(m["preview"])>220 else m["preview"],
        })
    df_hits = pd.DataFrame(rows)
    
    return df_hits, contexts

In [ ]:
QUERY = "Elizabeth Bennet's changing feelings for Mr. Darcy"
# contexts: Collection of source documents or chunks
df_hits, contexts = search_windowed(QUERY, topk=TOPK, max_out=8)

print("=== Initial ANN Hits ===")
display(df_hits)

print("\n=== Windowed Contexts (merged) ===")
display(pd.DataFrame([{
    "title": c["title"],
    "source": Path(c["source_path"]).name,
    "span": f"{c['start_chunk']}–{c['end_chunk']}",
    "approx_words": c["approx_words"],
    "span_count": c["span_count"],
    "preview": (c["text"][:220] + "…") if len(c["text"]) > 220 else c["text"]
} for c in contexts]))